In [33]:
!pip install setuptools==65.5.1 --quiet # Downgrade setuptools to resolve torch conflict
!pip install --upgrade --force-reinstall alpaca-py pandas --quiet # Removed trading-calendars

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 16.0 which is incompatible.
gradio-client 1.14.0 requires websockets<16.0,>=13.0, but you have websockets 16.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [37]:
import os
from google.colab import userdata
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
import alpaca.data.timeframe as timeframe_module  # Importing module with alias
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
import pandas as pd
from datetime import datetime, timedelta

In [6]:
# Configuración de entorno (Paper Trading para pruebas)
PAPER_TRADING = True
BASE_URL = "https://paper-api.alpaca.markets"

try:
    API_KEY = userdata.get('ALPACA_KEY')
    SECRET_KEY = userdata.get('ALPACA_SECRET')
except Exception:
    # Por si prefieres hardcodearlas (no recomendado si compartes el notebook)
    API_KEY = "TU_API_KEY_AQUI"
    SECRET_KEY = "TU_SECRET_KEY_AQUI"

# Inicializar clientes de Alpaca
trading_client = TradingClient(API_KEY, SECRET_KEY, paper=PAPER_TRADING)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

print("✅ Clientes de Alpaca inicializados correctamente.")

✅ Clientes de Alpaca inicializados correctamente.


### El Algoritmo de Dip Buying

Este script analizará los activos elegidos (SPY, QQQ, AAPL, MSFT), buscará caídas intradía de entre el 3% y el 7% comparado con el cierre del día anterior, y colocará una orden de compra con una orden limitada de venta automática (Take Profit) del 3% para buscar ese rebote rápido.

In [7]:
ACTIVOS = ["SPY", "QQQ", "AAPL", "MSFT"]
MIN_DROP = 0.03  # 3% mínimo de caída
MAX_DROP = 0.07  # 7% máximo (si cae más, puede ser una mala noticia grave)
TAKE_PROFIT_PCT = 0.03  # Objetivo de rebote del 3%
MONTO_POR_TRADE = 1000  # Dinero en USD a invertir por oportunidad

In [38]:
def ejecutar_dip_buying():
    print(f"🔍 Iniciando escaneo de mercado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    # 1. Obtener datos históricos recientes (Ayer y Hoy)
    # Usamos un rango de 3 días para asegurar que atrapamos el cierre anterior
    start_date = datetime.now() - timedelta(days=3)

    request_params = StockBarsRequest(
        symbol_or_symbols=ACTIVOS,
        timeframe=timeframe_module.TimeFrame.Day, # Using explicit module alias
        start=start_date
    )

    bars = data_client.get_stock_bars(request_params)
    df = bars.df

    for simbolo in ACTIVOS:
        if simbolo in df.index.get_level_values('symbol'): # Corrected line
            try:
                datos_activo = df.xs(simbolo)
                if len(datos_activo) < 2:
                    continue

                precio_cierre_ayer = datos_activo['close'].iloc[-2]
                precio_actual = datos_activo['close'].iloc[-1]
                volumen_hoy = datos_activo['volume'].iloc[-1]
                volumen_medio = datos_activo['volume'].rolling(window=2, min_periods=1).mean().iloc[-1]

                # Calcular variación porcentual del día
                variacion = (precio_actual - precio_cierre_ayer) / precio_cierre_ayer

                print(f"📊 {simbolo}: Precio actual: ${precio_actual:.2f} | Variación: {variacion*100:.2f}%")

                # 2. Verificar condiciones (Filtro de Caída y Volumen Alto)
                # Nota: variacion es negativa en caídas, por eso usamos valores negativos
                if -MAX_DROP <= variacion <= -MIN_DROP:

                    # Verificación básica de volumen alto (mayor al promedio reciente)
                    if volumen_hoy > (volumen_medio * 0.9):
                        print(f"⚠️ ¡ALERTA COMPRA! {simbolo} ha caído un {abs(variacion)*100:.2f}% con volumen.")

                        # Comprobar si ya tenemos posición para no duplicar
                        posiciones = trading_client.get_all_positions()
                        ya_comprado = any(p.symbol == simbolo for p in posiciones)

                        if not ya_comprado:
                            # Calcular cantidad de acciones a comprar
                            qty_a_comprar = MONTO_POR_TRADE / precio_actual

                            # 3. Enviar Orden de Compra (Mercado)
                            order_data = MarketOrderRequest(
                                symbol=simbolo,
                                qty=round(qty_a_comprar, 4), # Alpaca soporta acciones fraccionadas en Paper
                                side=OrderSide.BUY,
                                time_in_force=TimeInForce.DAY
                            )

                            orden_compra = trading_client.submit_order(order_data)
                            print(f"🛒 Orden de COMPRA enviada para {simbolo}. Cantidad: {qty_a_comprar:.4f}")

                            # 4. Configurar Salida: Orden de Venta Limitada (Take Profit)
                            precio_target = precio_actual * (1 + TAKE_PROFIT_PCT)

                            orden_venta_data = LimitOrderRequest(
                                symbol=simbolo,
                                qty=round(qty_a_comprar, 4),
                                side=OrderSide.SELL,
                                limit_price=round(precio_target, 2),
                                time_in_force=TimeInForce.GTC # Good 'Til Cancelled (Hasta que se ejecute o cancele)
                            )

                            orden_venta = trading_client.submit_order(orden_venta_data)
                            print(f"🎯 Orden de VENTA (Take Profit) colocada a ${precio_target:.2f}")
                        else:
                            print(f"Omitiendo: Ya tienes una posición abierta en {simbolo}.")
                    else:
                        print(f"❌ {simbolo} cayó, pero no cumple con el filtro de volumen alto.")

            except Exception as e:
                print(f"❌ Error procesando {simbolo}: {e}")

In [36]:
# Ejecutar la función
ejecutar_dip_buying()

🔍 Iniciando escaneo de mercado: 2026-05-27 13:26:37

